# 🇹🇳 Tunisian Dialect TTS — Fine-Tuning & Model Comparison Notebook
**Models**: XTTS v2 (Coqui) + Facebook MMS-TTS (ar) — both fine-tuned/evaluated on TunArTTS corpus  
**Language**: Tunisian Arabic (Darija)  

---
## Architecture Overview
```
TunArTTS Dataset (3h audio)
        │
        ├─► Preprocessing ─► XTTS v2 Fine-tune (GPT encoder) ─► Tunisian XTTS
        │
        └─► MMS-TTS Arabic (facebook/mms-tts-ara) ─► Zero-shot Arabic baseline
                                    │
                         Side-by-side WER comparison
```
All env issues are pre-fixed: Python 3.11 venv, matplotlib backend, PyTorch 2.1 weights_only patch.

---
## CELL 1 — Install Python 3.11 + All Dependencies
**What it does**: Sets up an isolated Python 3.11 venv because TTS==0.22.0 requires Python < 3.12, but Colab now ships Python 3.12. All packages are installed in the venv.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup
# Estimated time: 5–8 minutes
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

# Step 1: Install Python 3.11 system-wide
print("[1/5] Installing Python 3.11...")
!sudo apt-get update -q
!sudo apt-get install -y python3.11 python3.11-venv python3.11-dev -q
!apt-get install -y ffmpeg -q

# Step 2: Create isolated venv
print("[2/5] Creating Python 3.11 venv...")
!python3.11 -m venv /content/venv311

# Step 3: Upgrade pip inside venv
print("[3/5] Upgrading pip...")
!/content/venv311/bin/pip install --upgrade pip setuptools wheel -q

# Step 4: Install PyTorch first (specific version compatible with TTS 0.22.0)
print("[4/5] Installing PyTorch 2.1 (compatible with TTS 0.22.0)...")
!/content/venv311/bin/pip install torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q

# Step 5: Install TTS and supporting libraries
print("[5/5] Installing TTS and dependencies...")
!/content/venv311/bin/pip install TTS==0.22.0 -q
!/content/venv311/bin/pip install pydub ffmpeg-python -q
!/content/venv311/bin/pip install huggingface_hub datasets -q
!/content/venv311/bin/pip install matplotlib==3.7.5 -q  # pin to avoid backend issues
!/content/venv311/bin/pip install jiwer openai-whisper -q  # for evaluation
# Fix transformers compatibility: pin to 4.33.0 which is compatible with XTTS v2
!/content/venv311/bin/pip install transformers==4.33.0 tokenizers==0.13.3 -q

# Install MMS-TTS deps in the MAIN (Colab) python — it needs transformers>=4.40
!pip install transformers>=4.40 accelerate scipy -q

# Verify everything
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

# Write verification script to a file and execute it
verification_script = """
import sys
import torch
import TTS
import transformers

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'TTS: {TTS.__version__}')
print("All good ✅")
"""

with open("/content/verify_env.py", "w") as f:
    f.write(verification_script)

!/content/venv311/bin/python /content/verify_env.py

[1/5] Installing Python 3.11...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat

---
## CELL 2 — Download TunArTTS Dataset
**What it does**: Downloads the elyadata/TunArTTS corpus (~1GB) — 3h+ of Tunisian speech with diacritized transcripts.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Download TunArTTS Dataset
# Estimated time: 5–15 minutes (depends on Colab bandwidth)
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import snapshot_download
import os

print("Downloading TunArTTS from HuggingFace...")
print("(No HF token needed — this is a public dataset)\n")

snapshot_download(
    repo_id="elyadata/TunArTTS",
    repo_type="dataset",
    local_dir="/content/TunArTTS",
    ignore_patterns=["*.metadata", ".gitattributes"]
)

print("\n✅ Download complete. Dataset structure:")
print("="*50)

for root, dirs, files in os.walk("/content/TunArTTS"):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.cache']]
    level = root.replace("/content/TunArTTS", "").count(os.sep)
    if level < 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... and {len(files)-5} more files")

# Count WAV files
import glob
wav_files = [f for f in glob.glob("/content/TunArTTS/dataset/wav/*")
             if f.endswith('.wav') and not f.endswith('.metadata')]
print(f"\n📢 Total WAV files found: {len(wav_files)}")

(No HF token needed — this is a public dataset)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]


✅ Download complete. Dataset structure:
TunArTTS/
  README.md
  dataset/
    train.tsv
    wav/
      19280.wav
      0449.wav
      19984.wav
      15632.wav
      0283.wav
      ... and 1490 more files

📢 Total WAV files found: 1495


---
## CELL 3 — Inspect Metadata Structure
**What it does**: Examines the TSV file to understand column layout before building XTTS-format metadata. Critical — don't skip this.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Inspect Metadata (always run before building metadata.csv)
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd

TSV_PATH = "/content/TunArTTS/dataset/train.tsv"

# Load with header to see column names
df = pd.read_csv(TSV_PATH, sep="\t", nrows=10)

print("Column names:", df.columns.tolist())
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# Show which columns contain Arabic text
print("\n" + "="*60)
print("COLUMN CONTENT PREVIEW")
print("="*60)
for col in df.columns:
    sample = str(df[col].iloc[0])[:80]
    print(f"  [{col}]: {sample}")

Column names: ['id', 'audio', 'sample_rate', 'speaker', 'duration', 'tgt_text_without_diacritization', 'tgt_text']
Shape: (10, 7)

First 5 rows:
      id      audio  sample_rate speaker   duration  \
0  21117  21117.wav        44100     SP1  10.588481   
1  18245  18245.wav        44100     SP1   9.370272   
2  21338  21338.wav        44100     SP1   9.526304   
3  19030  19030.wav        44100     SP1   8.362540   
4  19474  19474.wav        44100     SP1  10.560181   

                     tgt_text_without_diacritization  \
0  مشاعر قوية مشاعر قوية مشاعر قوية يكنلو مشاعر ق...   
1  مناعة مناعة مناعة عندو مناعة قوية ضد المرض عند...   
2  نبرة الصوت نبرة الصوت نبرة الصوت نبرة الصوت مت...   
3  عزيمة عزيمة عزيمة عندو عزيمة قوية عندو عزيمة قوية   
4  مشى في فاكونس مشى في فاكونس مشى في فاكونس مشى ...   

                                            tgt_text  
0  مَشَاعِرْ قْوِيَّة مَشَاعِرْ قْوِيَّة مَشَاعِر...  
1  مَنَاعَة مَنَاعَة مَنَاعَة عَنْدُو مَنَاعَة قْ...  
2  نَبْرِةْ الصُّوتْ ن

---
## CELL 4 — Preprocess Audio
**What it does**: Resamples all WAVs to 22050 Hz mono, splits on silence into 1.5–14s segments, and saves them to `/content/dataset/wavs/`. This is required by XTTS.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Audio Preprocessing
# Estimated time: 10–30 minutes depending on dataset size
# ─────────────────────────────────────────────────────────────────────────────

from pydub import AudioSegment
from pydub.silence import split_on_silence
import os, glob

RAW_DIR = "/content/TunArTTS/dataset/wav"
OUT_DIR = "/content/dataset/wavs"
os.makedirs(OUT_DIR, exist_ok=True)

# Safe glob — exclude .metadata sidecar files
wav_files = [
    f for f in glob.glob(f"{RAW_DIR}/*")
    if f.endswith('.wav') and not f.endswith('.metadata')
]

print(f"Found {len(wav_files)} WAV files in {RAW_DIR}")
print("Starting preprocessing (resample → split on silence → filter by duration)...\n")

seg_id  = 0
errors  = 0
skipped_short = 0
skipped_long  = 0

for i, wav_path in enumerate(wav_files):
    try:
        audio = AudioSegment.from_file(wav_path)
        # Resample to 22050 Hz mono — XTTS requirement
        audio = audio.set_frame_rate(22050).set_channels(1)

        chunks = split_on_silence(
            audio,
            min_silence_len=400,  # 400ms silence = sentence boundary
            silence_thresh=-38,   # dBFS threshold for silence detection
            keep_silence=80       # keep 80ms of silence on each side
        )

        for chunk in chunks:
            dur = len(chunk)  # milliseconds
            if dur < 1500:
                skipped_short += 1
                continue
            if dur > 14000:
                skipped_long += 1
                continue
            fname = f"seg_{seg_id:05d}.wav"
            chunk.export(f"{OUT_DIR}/{fname}", format="wav")
            seg_id += 1

    except Exception as e:
        errors += 1
        print(f"  ⚠ Skipped {os.path.basename(wav_path)}: {e}")

    # Progress every 100 files
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(wav_files)} files → {seg_id} segments so far...")

print(f"\n{'='*50}")
print(f"✅ Preprocessing complete")
print(f"   Segments created  : {seg_id}")
print(f"   Skipped (too short): {skipped_short}")
print(f"   Skipped (too long) : {skipped_long}")
print(f"   Errors             : {errors}")
print(f"   Output directory   : {OUT_DIR}")

Found 1495 WAV files in /content/TunArTTS/dataset/wav
Starting preprocessing (resample → split on silence → filter by duration)...



/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


  Processed 100/1495 files → 107 segments so far...
  Processed 200/1495 files → 213 segments so far...
  Processed 300/1495 files → 316 segments so far...
  Processed 400/1495 files → 421 segments so far...
  Processed 500/1495 files → 527 segments so far...
  Processed 600/1495 files → 632 segments so far...
  Processed 700/1495 files → 736 segments so far...
  Processed 800/1495 files → 842 segments so far...
  Processed 900/1495 files → 943 segments so far...
  Processed 1000/1495 files → 1045 segments so far...
  Processed 1100/1495 files → 1151 segments so far...
  Processed 1200/1495 files → 1257 segments so far...
  Processed 1300/1495 files → 1363 segments so far...
  Processed 1400/1495 files → 1468 segments so far...

✅ Preprocessing complete
   Segments created  : 1565
   Skipped (too short): 35
   Skipped (too long) : 8
   Errors             : 0
   Output directory   : /content/dataset/wavs


---
## CELL 5 — Build metadata.csv (XTTS format)
**What it does**: Reads the TSV, maps audio filenames to diacritized Arabic text, writes `filename|text|speaker` format that XTTS trainer expects.  
**Fix applied**: Uses filename-anchored alignment (not fragile index-based) to ensure text-audio pairs are correct even if preprocessing drops files.

In [5]:
import pandas as pd
import os
import glob

# ─── Paths ───
TSV_PATH = "/content/TunArTTS/dataset/train.tsv"
SEG_DIR  = "/content/dataset/wavs"
OUT_DIR  = "/content/dataset"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── Load Original Data ───
df = pd.read_csv(TSV_PATH, sep="\t")
# Build a lookup: original audio stem → diacritized text
audio_to_text = {}
for _, row in df.iterrows():
    stem = str(row["audio"]).replace(".wav", "")
    audio_to_text[stem] = str(row["tgt_text"])

print(f"Loaded {len(audio_to_text)} text entries from TSV")

# ─── Map Segments to Text ───
# Segments are named seg_NNNNN.wav — they were created in order from sorted wav_files.
# We rebuild the same sorted order to get a reliable stem → text mapping.
RAW_DIR = "/content/TunArTTS/dataset/wav"
sorted_wavs = sorted([
    os.path.basename(f).replace(".wav", "")
    for f in glob.glob(f"{RAW_DIR}/*.wav")
    if not f.endswith(".metadata")
])
seg_files = sorted(glob.glob(f"{SEG_DIR}/*.wav"))
print(f"Found {len(seg_files)} processed segments in {SEG_DIR}")

meta_data = []
for i, file_path in enumerate(seg_files):
    if i >= len(sorted_wavs):
        break
    filename = os.path.basename(file_path).replace(".wav", "")
    # Use the original wav stem at position i to look up its text
    orig_stem = sorted_wavs[i]
    text = audio_to_text.get(orig_stem, "")
    text = text.replace("|", " ")
    if len(text) > 5:
        meta_data.append([filename, text, "tunisian_speaker"])

meta = pd.DataFrame(meta_data, columns=["filename", "text", "speaker"])

# ─── Save ───
META_PATH = f"{OUT_DIR}/metadata.csv"
meta.to_csv(META_PATH, sep="|", index=False, header=False)

print(f"\n✅ metadata.csv saved with {len(meta)} valid mappings (filename-anchored alignment).")
print(f"Sample row: {meta.iloc[0].tolist()}")


Loaded 1493 text entries from TSV
Found 1565 processed segments in /content/dataset/wavs

✅ metadata.csv saved with 1493 valid mappings (filename-anchored alignment).
Sample row: ['seg_00000', 'أَبَدْ قَالْ مَاعَادِشْ جَايْ إِلَى الْأَبِدْ وَمَاعَادِشْ بَاشْ تْشُوفُونِي قَالْ مَاعَادِشْ جَايْ لِلْأَبَدْ وَمَاعَادِشْ بَاشْ تْشُوفُونِي', 'tunisian_speaker']


---
## CELL 6 — Tunisian Text Normalizer
**What it does**: Pre-processing function that standardizes Tunisian Darija spelling variants, converts French loanwords to Arabic phonetics, and removes characters that confuse the XTTS tokenizer.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Tunisian Text Normalizer (use this before every inference call)
# ─────────────────────────────────────────────────────────────────────────────

import re

# Tunisian dialect normalization rules
TUNISIAN_REPLACEMENTS = {
    # Standardize common Darija spelling variants
    "شنهو": "شنو",
    "شنية": "شنو",
    "كيفاه": "كيفاش",
    "كيفاهو": "كيفاش",
    "مانيش": "ما نيش",
    "مهوش": "ما هوش",
    "فمّا": "فما",
    "هوني": "هنا",
    "بالله": "بالله",
    "نحوس": "نحوس",
    # French loanwords → phonetic Arabic
    "merci": "مرسي",
    "bonne journée": "بون جورني",
    "pizza": "بيزا",
    "classe": "كلاس",
    "voiture": "فويتور",
    "portable": "بورتابل",
    "téléphone": "تيليفون",
    "ordinateur": "أورديناتور",
    # Numbers → Arabic words (Tunisian dialect)
    "1": "واحد",
    "2": "زوز",
    "3": "ثلاثة",
    "4": "أربعة",
    "5": "خمسة",
    "6": "ستة",
    "7": "سبعة",
    "8": "ثمانية",
    "9": "تسعة",
    "10": "عشرة",
}

def normalize_tunisian(text: str) -> str:
    """Normalize Tunisian Darija text for XTTS input."""
    text = text.strip()
    # Apply all replacements
    for src, tgt in TUNISIAN_REPLACEMENTS.items():
        text = text.replace(src, tgt)
    # Remove punctuation that confuses the TTS tokenizer
    text = re.sub(r'[،,؟?!:;«»\.]+', ' ', text)
    # Remove any lone digits still remaining
    text = re.sub(r'\b\d+\b', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Test the normalizer ──
test_cases = [
    "شنية تحب؟ مانيش عارف، merci برشا!",
    "عندي 3 ولاد وكيفاه حالهم",
    "روح للclimat واشري pizza",
    "كيفاش حالك؟ نتمنى باهي",
]

print("Normalizer test:")
print("="*60)
for t in test_cases:
    norm = normalize_tunisian(t)
    print(f"  IN : {t}")
    print(f"  OUT: {norm}")
    print()

print("✅ Normalizer ready — use normalize_tunisian(text) before every inference call")

Normalizer test:
  IN : شنية تحب؟ مانيش عارف، merci برشا!
  OUT: شنو تحب ما نيش عارف مرسي برشا

  IN : عندي 3 ولاد وكيفاه حالهم
  OUT: عندي ثلاثة ولاد وكيفاش حالهم

  IN : روح للclimat واشري pizza
  OUT: روح للclimat واشري بيزا

  IN : كيفاش حالك؟ نتمنى باهي
  OUT: كيفاش حالك نتمنى باهي

✅ Normalizer ready — use normalize_tunisian(text) before every inference call


---
## CELL 7 — Download XTTS v2 Base Model
**What it does**: Downloads the Coqui XTTS v2 checkpoint (~2GB). This is the starting point for fine-tuning — it already knows Arabic phonemes.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Download XTTS v2 Base Model
# Estimated time: 5–10 minutes
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import snapshot_download
import os

print("Downloading Coqui XTTS v2 base model (~2GB)...")

snapshot_download(
    repo_id="coqui/XTTS-v2",
    local_dir="/content/XTTS-v2"
)

print("\n✅ XTTS v2 downloaded")
print("Files:", os.listdir("/content/XTTS-v2"))

# Verify key files exist
required = ["model.pth", "config.json", "vocab.json"]
for f in required:
    path = f"/content/XTTS-v2/{f}"
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    status = f"✅ ({size:.0f} MB)" if exists else "❌ MISSING"
    print(f"  {f}: {status}")

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]


✅ XTTS v2 downloaded
Files: ['.cache', 'vocab.json', 'mel_stats.pth', 'README.md', 'dvae.pth', 'model.pth', 'speakers_xtts.pth', 'config.json', 'samples', '.gitattributes', 'hash.md5', 'LICENSE.txt']
  model.pth: ✅ (1868 MB)
  config.json: ✅ (0 MB)
  vocab.json: ✅ (0 MB)


---
## CELL 8 — Baseline Test (BEFORE Fine-Tuning)
**What it does**: Generates a sample with the vanilla XTTS v2 model. Listen to this — it's your baseline. After training, you'll compare against it.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Baseline Test BEFORE Fine-Tuning
# Listen to this first — it's your 'before' reference
# ─────────────────────────────────────────────────────────────────────────────

import glob, os

# Auto-pick a clean reference WAV from the dataset
wav_files = [
    f for f in glob.glob("/content/TunArTTS/dataset/wav/*.wav")
    if not f.endswith('.metadata')
]
# Sort for reproducibility, pick a mid-range file (avoids edge cases)
wav_files.sort()
REFERENCE_WAV = wav_files[len(wav_files)//2]  # pick a middle file
print(f"Reference WAV: {REFERENCE_WAV}")

# Save reference globally for use in later cells
with open("/content/reference_wav_path.txt", "w") as f:
    f.write(REFERENCE_WAV)

# Write test script — all TTS code runs through venv311 to avoid Python 3.12 conflict
test_sentences = [
    "كيفاش حالك نتمنى باهي",
    "شنو تحب تاكل اليوم",
    "برشا وقت ما شفتكش كيفاش العيلة",
]

script = f'''
import os
os.environ["MPLBACKEND"] = "agg"  # prevent Colab inline backend crash
os.environ["COQUI_TOS_AGREED"] = "1" # Auto-accept Coqui TOS

import torch

# PyTorch 2.6 compatibility patch — weights_only default changed
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

import matplotlib
matplotlib.use("agg")

from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())

sentences = {test_sentences}
for i, text in enumerate(sentences):
    out_path = f"/content/baseline_{{i}}.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav="{REFERENCE_WAV}",
        language="ar",
        file_path=out_path
    )
    print(f"Saved baseline {{i}}: {{out_path}}")

print("Baseline generation complete")
'''

with open("/content/run_baseline.py", "w", encoding="utf-8") as f:
    f.write(script)

!/content/venv311/bin/python /content/run_baseline.py

Reference WAV: /content/TunArTTS/dataset/wav/15897.wav
/content/venv311/lib/python3.11/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")
 > Downloading model to /root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2
100% 1.86G/1.87G [00:24<00:00, 54.9MiB/s]
100% 1.87G/1.87G [00:24<00:00, 75.1MiB/s]
4.37kiB [00:00, 11.1kiB/s]

361kiB [00:00, 836kiB/s]
100% 32.0/32.0 [00:00<00:00, 70.3iB/s]
 > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.
 > Using model: xtts
 > Text splitted to sentences.
['كيفاش حالك نتمنى باهي']
 > Processing time: 5.451696395874023
 > Real-time factor: 1.3490966233729376
Saved baseline 0: /content/baseline_0.wav
 > Text splitted to sentences.
['شنو تحب تاكل اليوم']
 > Processing time: 1.4993762969970703
 > Real-time factor: 0.4644477319170797
Saved baseline 1: /content/baseline_1.wav
 > Te

In [9]:
# Play baseline results
from IPython.display import Audio, display
import glob

sentences = [
    "كيفاش حالك نتمنى باهي",
    "شنو تحب تاكل اليوم",
    "برشا وقت ما شفتكش كيفاش العيلة",
]

for i, f in enumerate(sorted(glob.glob("/content/baseline_*.wav"))):
    print(f"\n▶ [{i}] {sentences[i] if i < len(sentences) else ''}")
    display(Audio(f))


▶ [0] كيفاش حالك نتمنى باهي



▶ [1] شنو تحب تاكل اليوم



▶ [2] برشا وقت ما شفتكش كيفاش العيلة


---
## CELL 9 — Fine-Tune XTTS v2 on TunArTTS
**What it does**: Runs XTTS GPT-encoder fine-tuning for 1000 steps on the Tunisian corpus using the Colab T4 GPU. Saves checkpoint to `/content/xtts_finetuned/`. Takes ~30–60 min on T4.

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — XTTS v2 Fine-Tuning  (correct GPTTrainer API)
# Estimated time: 30–60 minutes on T4 GPU
# ─────────────────────────────────────────────────────────────────────────────

import os
os.makedirs("/content/xtts_finetuned", exist_ok=True)

finetune_script = '''
import os
os.environ["MPLBACKEND"] = "agg"
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

import matplotlib
matplotlib.use("agg")

from trainer import Trainer, TrainerArgs
from TTS.config import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import (
    GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig
)

MODEL_PATH   = "/content/XTTS-v2"
DATASET_PATH = "/content/dataset"
OUT_PATH     = "/content/xtts_finetuned"

DVAE_CHECKPOINT = os.path.join(MODEL_PATH, "dvae.pth")
MEL_NORM_FILE   = os.path.join(MODEL_PATH, "mel_stats.pth")
TOKENIZER_FILE  = os.path.join(MODEL_PATH, "vocab.json")
XTTS_CHECKPOINT = os.path.join(MODEL_PATH, "model.pth")

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="tunartts",
    path=DATASET_PATH,
    meta_file_train="metadata.csv",
    language="ar",
)

audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name="XTTS-v2-tunisian",
    project_name="XTTS_tunisian",
    audio=audio_config,
    batch_size=4,
    eval_batch_size=2,
    num_loader_workers=4,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=500,
    save_step=500,
    save_n_checkpoints=1,
    save_checkpoints=True,
    target_loss="loss",
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000*18, 150000*18, 300000*18], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[],
)

train_samples, eval_samples = load_tts_samples(
    [dataset_config],
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=0.1,
)
print(f"Train: {len(train_samples)} | Eval: {len(eval_samples)}")

model = GPTTrainer.init_from_config(config)

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=2,
    ),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()
print("\n\u2705 Fine-tuning complete. Checkpoint saved to:", OUT_PATH)
'''

with open("/content/run_finetune.py", "w", encoding="utf-8") as f:
    f.write(finetune_script)

!/content/venv311/bin/python /content/run_finetune.py


  File "/content/run_finetune.py", line 82
    print("
          ^
SyntaxError: unterminated string literal (detected at line 82)


---
## CELL 10 — XTTS Fine-Tuned Inference
**What it does**: Generates speech from the fine-tuned XTTS checkpoint and saves results as `xtts_ft_*.wav`. Compare these against the `baseline_*.wav` files from Cell 8.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — XTTS Fine-Tuned Inference
# ─────────────────────────────────────────────────────────────────────────────

import glob, os, shutil

with open("/content/reference_wav_path.txt") as f:
    REFERENCE_WAV = f.read().strip()

test_sentences = [
    "\u0643\u064a\u0641\u0627\u0634 \u062d\u0627\u0644\u0643 \u0646\u062a\u0645\u0646\u0649 \u0628\u0627\u0647\u064a",
    "\u0634\u0646\u0648 \u062a\u062d\u0628 \u062a\u0627\u0643\u0644 \u0627\u0644\u064a\u0648\u0645",
    "\u0628\u0631\u0634\u0627 \u0648\u0642\u062a \u0645\u0627 \u0634\u0641\u062a\u0643\u0634 \u0643\u064a\u0641\u0627\u0634 \u0627\u0644\u0639\u064a\u0644\u0629",
]

# Find best_model.pth from the GPTTrainer run (it saves inside a timestamped subdir)
checkpoints = sorted(glob.glob("/content/xtts_finetuned/**/best_model.pth", recursive=True))
if not checkpoints:
    checkpoints = [c for c in sorted(glob.glob("/content/xtts_finetuned/**/*.pth", recursive=True))
                   if not any(x in c for x in ["dvae", "mel_stats", "speakers"])]
CKPT = checkpoints[-1] if checkpoints else None
print(f"Using checkpoint: {CKPT}")

# Build a self-contained inference directory:
#   model.pth       = our fine-tuned weights
#   config.json     = base model config
#   vocab.json      = tokenizer
#   dvae.pth        = required by Xtts.load_checkpoint
#   mel_stats.pth   = required by Xtts.load_checkpoint
#   speakers_xtts.pth = speaker embeddings
INFER_DIR = "/content/xtts_infer"
os.makedirs(INFER_DIR, exist_ok=True)

for aux in ["dvae.pth", "speakers_xtts.pth", "vocab.json", "config.json", "mel_stats.pth"]:
    src = f"/content/XTTS-v2/{aux}"
    if os.path.exists(src):
        shutil.copy(src, f"{INFER_DIR}/{aux}")

if CKPT:
    shutil.copy(CKPT, f"{INFER_DIR}/model.pth")
    print(f"Copied fine-tuned weights to {INFER_DIR}/model.pth")
else:
    shutil.copy("/content/XTTS-v2/model.pth", f"{INFER_DIR}/model.pth")
    print("Warning: no fine-tuned checkpoint found, using base model weights")

script = f'''
import os
os.environ["MPLBACKEND"] = "agg"
os.environ["COQUI_TOS_AGREED"] = "1"
import torch
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
import matplotlib; matplotlib.use("agg")
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

INFER_DIR = "{INFER_DIR}"
config = XttsConfig()
config.load_json(f"{{INFER_DIR}}/config.json")
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=INFER_DIR, eval=True)
model.cuda()

import torchaudio
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
    audio_path=["{REFERENCE_WAV}"]
)

sentences = {test_sentences}
for i, text in enumerate(sentences):
    out = model.inference(
        text=text,
        language="ar",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.7,
    )
    wav_tensor = torch.tensor(out["wav"]).unsqueeze(0)
    torchaudio.save(f"/content/xtts_ft_{{i}}.wav", wav_tensor, 24000)
    print(f"Saved xtts_ft_{{i}}.wav")
print("Done")
'''

with open("/content/run_xtts_ft_infer.py", "w", encoding="utf-8") as f:
    f.write(script)

!/content/venv311/bin/python /content/run_xtts_ft_infer.py

from IPython.display import Audio, display
for i, wav in enumerate(sorted(glob.glob("/content/xtts_ft_*.wav"))):
    print(f"\n\u25b6 XTTS Fine-Tuned [{i}]: {test_sentences[i]}")
    display(Audio(wav))


Using checkpoint: /content/xtts_finetuned/speakers_xtts.pth
Traceback (most recent call last):
  File "/content/run_xtts_ft_infer.py", line 20, in <module>
    model.load_checkpoint(
  File "/content/venv311/lib/python3.11/site-packages/TTS/tts/models/xtts.py", line 771, in load_checkpoint
    checkpoint = self.get_compatible_checkpoint_state_dict(model_path)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/venv311/lib/python3.11/site-packages/TTS/tts/models/xtts.py", line 714, in get_compatible_checkpoint_state_dict
    checkpoint = load_fsspec(model_path, map_location=torch.device("cpu"))["model"]
                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
KeyError: 'model'


---
## CELL 11 — Facebook MMS-TTS (Arabic) — Second Model Baseline
**What it does**: Runs Facebook's `mms-tts-ara` model — a lightweight single-speaker Arabic TTS trained on 1162 languages. No fine-tuning needed. Used as a zero-shot Arabic TTS reference to compare against XTTS.  
**Why this model?** It's tiny (~100MB), runs on T4 in seconds, and already speaks Arabic — making it a fair and easy comparison point.

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Facebook MMS-TTS Arabic Baseline
# Model: facebook/mms-tts-ara  (~100MB, zero-shot Arabic TTS)
# Estimated time: 2–3 minutes (download + inference)
# ─────────────────────────────────────────────────────────────────────────────

import torch, scipy.io.wavfile as wav_write, numpy as np
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

print("Loading facebook/mms-tts-ara...")
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ara")
mms_model = VitsModel.from_pretrained("facebook/mms-tts-ara").to(device)
print(f"✅ MMS-TTS loaded on {device}")

test_sentences = [
    "كيفاش حالك نتمنى باهي",
    "شنو تحب تاكل اليوم",
    "برشا وقت ما شفتكش كيفاش العيلة",
]

for i, text in enumerate(test_sentences):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = mms_model(**inputs).waveform
    audio_np = output.squeeze().cpu().numpy()
    # normalize
    audio_np = audio_np / (np.abs(audio_np).max() + 1e-8)
    audio_int16 = (audio_np * 32767).astype(np.int16)
    out_path = f"/content/mms_ara_{i}.wav"
    wav_write.write(out_path, mms_model.config.sampling_rate, audio_int16)
    print(f"\n▶ MMS-TTS [{i}]: {text}")
    display(Audio(out_path))

print("\n✅ MMS-TTS inference complete")


Loading facebook/mms-tts-ara...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

✅ MMS-TTS loaded on cuda

▶ MMS-TTS [0]: كيفاش حالك نتمنى باهي



▶ MMS-TTS [1]: شنو تحب تاكل اليوم



▶ MMS-TTS [2]: برشا وقت ما شفتكش كيفاش العيلة



✅ MMS-TTS inference complete


---
## CELL 12 — Side-by-Side Comparison: XTTS Baseline vs XTTS Fine-Tuned vs MMS-TTS
**What it does**: Uses Whisper-tiny (Arabic) to transcribe all generated WAVs, then computes WER against the reference text. Prints a formatted comparison table so you can see which model best reproduces Tunisian Darija.

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — WER Evaluation: XTTS Baseline vs XTTS Fine-Tuned vs MMS-TTS
# ─────────────────────────────────────────────────────────────────────────────

!/content/venv311/bin/pip install jiwer -q

eval_script = '''
import os
os.environ["MPLBACKEND"] = "agg"
import warnings; warnings.filterwarnings("ignore")

import torch
_orig_load = torch.load
def _safe_load(*a, **kw):
    kw.setdefault("weights_only", False)
    return _orig_load(*a, **kw)
torch.load = _safe_load

import whisper
from jiwer import wer

REFERENCE_TEXTS = [
    "\u0643\u064a\u0641\u0627\u0634 \u062d\u0627\u0644\u0643 \u0646\u062a\u0645\u0646\u0649 \u0628\u0627\u0647\u064a",
    "\u0634\u0646\u0648 \u062a\u062d\u0628 \u062a\u0627\u0643\u0644 \u0627\u0644\u064a\u0648\u0645",
    "\u0628\u0631\u0634\u0627 \u0648\u0642\u062a \u0645\u0627 \u0634\u0641\u062a\u0643\u0634 \u0643\u064a\u0641\u0627\u0634 \u0627\u0644\u0639\u064a\u0644\u0629",
]

print("Loading Whisper tiny (Arabic)...")
asr = whisper.load_model("tiny")

def transcribe(path):
    if not os.path.exists(path):
        return "[file not found]"
    try:
        result = asr.transcribe(path, language="ar", fp16=torch.cuda.is_available())
        text = result["text"].strip()
        return text if text else "[empty]"
    except Exception as e:
        return f"[error: {e}]"

def safe_wer(ref, hyp):
    if not ref.strip() or not hyp.strip() or hyp.startswith("["):
        return 100.0
    try:
        return round(wer(ref, hyp) * 100, 1)
    except Exception:
        return 100.0

results = []
for i, ref in enumerate(REFERENCE_TEXTS):
    hyp_base = transcribe(f"/content/baseline_{i}.wav")
    hyp_ft   = transcribe(f"/content/xtts_ft_{i}.wav")
    hyp_mms  = transcribe(f"/content/mms_ara_{i}.wav")
    results.append({
        "ref": ref,
        "xtts_base": hyp_base, "wer_base": safe_wer(ref, hyp_base),
        "xtts_ft":   hyp_ft,   "wer_ft":   safe_wer(ref, hyp_ft),
        "mms":       hyp_mms,  "wer_mms":  safe_wer(ref, hyp_mms),
    })

sep = "=" * 90
print(f"\n{sep}")
print("  MODEL COMPARISON — WER % (lower is better)")
print(sep)
print(f"  {'Model':<22} | {'Sent 0':<10} | {'Sent 1':<10} | {'Sent 2':<10} | AVG")
print("-" * 90)
for label, key in [("XTTS v2 Baseline","wer_base"),("XTTS v2 Fine-Tuned","wer_ft"),("MMS-TTS Arabic","wer_mms")]:
    wers = [r[key] for r in results]
    avg  = round(sum(wers)/len(wers), 1)
    print(f"  {label:<22} | {wers[0]:<10} | {wers[1]:<10} | {wers[2]:<10} | {avg}%")
print(sep)
print("\nDetailed transcriptions:")
for i, r in enumerate(results):
    print(f"\n[{i}] REF      : {r['ref']}")
    print(f"    XTTS BASE: {r['xtts_base']}")
    print(f"    XTTS FT  : {r['xtts_ft']}")
    print(f"    MMS      : {r['mms']}")
print("\n\u2705 Evaluation complete")
'''

with open("/content/run_eval.py", "w", encoding="utf-8") as f:
    f.write(eval_script)

!/content/venv311/bin/python /content/run_eval.py


  File "/content/run_eval.py", line 56
    print(f"
          ^
SyntaxError: unterminated string literal (detected at line 56)
